# 02 — Scoring Validation

Summarizes the three validation artifacts that turn hand-chosen weights into measurable claims: the no-leakage **backtest**, the weight **sensitivity** analysis, and the **fairness** audit. Reports are produced by `src/backtest.py`, `src/sensitivity.py`, and `src/fairness_audit.py` and narrated in [`docs/model_card.md`](../docs/model_card.md).

In [1]:
from pathlib import Path
import json
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROC = REPO / 'data' / 'processed'
load = lambda name: json.loads((PROC / name).read_text())

## Backtest — does a higher score predict a *future* health-based violation?
Cutoff and horizon are frozen; time-varying components are recomputed using only records on or before the cutoff (no leakage).

In [2]:
bt = load('backtest_report.json')
print(f"cutoff {bt['cutoff']}  +{bt['horizon_months']}m horizon")
print(f"base rate: {bt['base_rate']:.2%}  ({bt['n_positive']} of {bt['n_systems']:,})")
rows = []
for label, key in [('model score', 'model_score_asof'), ('baseline: prior violations', 'baseline_prior_violations'), ('baseline: population', 'baseline_population')]:
    m = bt[key]
    rows.append({'model': label, 'AUC': m['auc'], 'precision@100': m['precision_at_k']['100'], 'lift@100': m['lift_at_k']['100']})
pd.DataFrame(rows).set_index('model')

cutoff 2023-12-31  +24m horizon
base rate: 2.07%  (338 of 16,339)


,AUC,precision@100,lift@100
model,,,
model score,0.7416,0.20,9.67
baseline: prior violations,0.7351,0.22,10.63
baseline: population,0.7350,0.06,2.90


**Reading it honestly:** the score roughly *ties* a prior-violation-count baseline on this single outcome. Its value-add is combining compliance, enforcement, vulnerability, drought and funding signals with explainability and an equity lens — not out-predicting a violations counter.

## Sensitivity — are the rankings robust to the weight choices?
Monte Carlo perturbation of every weight by +/-20% (renormalized).

In [3]:
sn = load('sensitivity_report.json')
print(json.dumps({k: v for k, v in sn.items() if not isinstance(v, (list, dict))}, indent=2))
sn

{
  "n_trials": 500,
  "weight_perturbation": "+/-20%",
  "top_k": 100
}


{'n_trials': 500,
 'weight_perturbation': '+/-20%',
 'top_k': 100,
 'monte_carlo': {'spearman_rank_correlation': {'mean': 0.9938,
   'min': 0.9712,
   'p05': 0.9814},
  'top100_retention': {'mean': 0.9475, 'min': 0.85, 'p05': 0.89},
  'tier_change_rate': {'mean': 0.0, 'max': 0.0}},
 'component_influence_when_removed': {'compliance_risk_component': {'spearman_vs_base': 0.969,
   'top100_retained': 0.28},
  'enforcement_risk_component': {'spearman_vs_base': 0.9868,
   'top100_retained': 0.95},
  'vulnerability_component': {'spearman_vs_base': 0.5876,
   'top100_retained': 0.52},
  'drought_component': {'spearman_vs_base': 0.7825, 'top100_retained': 0.79},
  'funding_gap_component': {'spearman_vs_base': 1.0, 'top100_retained': 1.0},
  'small_system_component': {'spearman_vs_base': 0.9719,
   'top100_retained': 0.8},
  'data_quality_penalty': {'spearman_vs_base': 0.9998,
   'top100_retained': 0.98}}}

## Fairness — is the score laundering demographics, or is the SVI tilt the intended equity weighting?

In [4]:
fr = load('fairness_report.json')
fr

{'note': 'SVI is an intended input (equity weighting) and a fairness axis. This tool is for directing support/funding, not enforcement targeting.',
 'correlations_with_svi': {'overall_score': 0.3429,
  'compliance_component_only': 0.0316,
  'score_excluding_vulnerability_input': -0.0967},
 'high_review_mean_svi': 0.4574,
 'overall_mean_svi': 0.3407,
 'by_svi_quartile': [{'quartile': 'Q1 (least)',
   'systems': 4136,
   'high_review_rate': 0.0039,
   'mean_score': 22.9},
  {'quartile': 'Q2',
   'systems': 4458,
   'high_review_rate': 0.0076,
   'mean_score': 24.71},
  {'quartile': 'Q3',
   'systems': 3685,
   'high_review_rate': 0.0152,
   'mean_score': 27.1},
  {'quartile': 'Q4 (most)',
   'systems': 3868,
   'high_review_rate': 0.0212,
   'mean_score': 31.46}]}

The objective compliance signal is ~uncorrelated with SVI, so the SVI tilt in the score is the transparent, intended equity weighting — which is why the tool is scoped to *support/funding*, not enforcement. See the model card for the full interpretation.